# Week 10 — Gap experiment and Phase 3 dataset generation

**Question.** Is a point that lies between two computed tori a non-torus point?
The naive labelling scheme for the classifier assumes yes. This notebook tests it.

**Label definition: C, reported as B.** The probe returns a continuous distance;
the binary label is a thresholded readout of it. Both are stored. Rationale in
`src/torus_probe.py` — briefly, a forward-time behavioural test is impossible here
because R_u = 146.32 gives only ~4 clean stroboscopic periods before integrator
error reaches O(1), which is one to two orders of magnitude short of what NAFF or
Das–Yorke weighted Birkhoff need. `converge_torus` never integrates longer than one
period, so the invariance equation is the only probe the shadowing bound permits.

**Scope.** Week 10 is dataset generation. This notebook ends at a saved labelled
dataset plus its threshold readout. No feature design, no training, no
phase-space failure map — those are Weeks 11–12.

**Runtime** ~5 min for the full sweep (80 Newton solves at N=31).

In [ ]:
import sys, time
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

import qpit
from cr3bp_3d import MU_EARTH_MOON as MU
from torus_probe import calibrate_probe, probe_gap, probe_curve, label_from_probe
from gap_sample import resonance_distance

LU = 384400.0
np.set_printoptions(precision=12)
t0 = time.time()

## 1. Load the family

`qpit_family_L2h29.npz` stores flat arrays; the probe wants records.

**The `argsort` is not decorative.** `C_A` happens to be stored ascending, which is
exactly the condition that makes a missing sort invisible — until a rerun with a
different continuation order pairs non-adjacent members and every gap silently
widens. Adjacency is the independent variable of this whole experiment.

In [ ]:
d = np.load('../data/qpit_family_L2h29.npz')

def records(prefix, arch):
    U, T, R, C, A = (arch[prefix + k] for k in ('_U', '_T', '_rho', '_C', '_A'))
    o = np.argsort(A)                       # see note above
    return [dict(U0=U[i], T=float(T[i]), rho=float(R[i]), C=float(C[i]),
                 amplitude=float(A[i]), idx=int(i)) for i in o]

famC = records('C', d)
kappa, drho_dC = float(d['kappa']), float(d['drho_halo_dC'])
print(f"famC {len(famC)} members   kappa={kappa:.6f}   drho_halo/dC={drho_dC:.6f}")

## 2. Census, and where the resonance actually is

Two things to read off.

**The Week 9 kappa predicts the family's own rho drift.** `kappa * A_max^2` should
reproduce the measured drift across the family. Expect `1.9884e-05` against a
measured `1.9862e-05` — four significant figures, and a fourth independent
validation of kappa.

**famC never approaches a resonance.** The drift is 2.0e-5; the nearest low-order
rational is 6.8e-3 away. Reaching it by amplitude needs ~9500 km, 18.5x beyond the
family's 515 km. Reaching it by *energy* needs dC ~ 1.7e-3, a routine step along
the halo family. That is the negative control, and it is a separate notebook.

In [ ]:
A = np.array([m['amplitude'] for m in famC])
Rn = np.array([m['rho'] for m in famC]) / (2 * np.pi)
gaps = np.diff(A) * LU

print(f"amplitude {A[0]*LU:.2f} -> {A[-1]*LU:.2f} km ; gaps {gaps.min():.2f} - {gaps.max():.2f} km")
print(f"rho/2pi drift measured : {Rn[0]-Rn[-1]:.4e}")
print(f"kappa * A_max^2        : {abs(kappa)*A[-1]**2:.4e}   <- Week 9 kappa, 4th validation")

for q_max in (7, 12):
    dres, pq = resonance_distance(famC[0]['rho'], q_max=q_max)
    A_res = np.sqrt(dres/abs(kappa))
    reach_amp = (pq[0]/pq[1] < Rn[0])       # kappa<0 drives rho DOWN
    print(f"\nnearest resonance q<={q_max:2d}: {pq[0]}/{pq[1]} = {pq[0]/pq[1]:.9f}  d = {dres:.4e}")
    print(f"  by amplitude : A = {A_res*LU:.0f} km ({A_res/A[-1]:.1f}x family)"
          f"   {'reachable' if reach_amp else 'WRONG DIRECTION (kappa<0)'}")
    print(f"  by energy    : dC = {(pq[0]/pq[1]-Rn[0])/drho_dC:+.6f}"
          f"  ->  C = {famC[0]['C']+(pq[0]/pq[1]-Rn[0])/drho_dC:.9f}")

## 3. Positive control and threshold calibration

**Perturb is not optional.** Re-probing a member from its own converged `U0` exits
after one Newton iteration with `dist_max` identically 0.0, and a zero floor times
any margin is still zero — every candidate in the experiment would be labelled
negative and it would look like a clean result confirming the naive scheme.
Displacing first makes the control an honest analogue of the treatment: a nearby
non-solution rather than an exact one.

Expect `dist_max` ~ 2.6e-6 to 3.8e-6, threshold ~ 3.8e-5.

In [ ]:
cal = calibrate_probe(famC[::4], MU, perturb=1e-6, offgrid=False)
print(f"n={cal['n_control']}  failed={cal['n_failed']}")
print("  dist_max per member:", " ".join(f"{p['dist_max']:.2e}" for p in cal['probes']))
print(f"  thresholds: dist_max<={cal['dist_max']:.3e}  damp_rel<={cal['damp_rel']:.3e}"
      f"  res_inf<={cal['res_inf']:.2e}    [{time.time()-t0:.0f}s]")

## 4. Treatment — every adjacent pair

**Prediction stated before running:** more than half of gap candidates lie on tori.

The interpolant is the only available construction. A gap point has no fixed point
to linearise about, so `seed_from_halo`'s route is unavailable there; fitting a
curve to a stroboscopic orbit would need >= N iterates against the ~4 the shadowing
bound allows. Two converged curves at 1e-12 are the only source of a full candidate
curve at gap amplitude.

In [ ]:
ALPHAS = [0.2, 0.4, 0.5, 0.6, 0.8]
curves = []
for k in range(len(famC) - 1):
    a, b = famC[k], famC[k+1]
    rows = probe_gap(a, b, MU, ALPHAS, offgrid=False)
    for p in rows:
        p.update(pair_lo=k, pair_hi=k+1, arm='gap',
                 gap_km=(b['amplitude']-a['amplitude'])*LU,
                 gap_rel=(b['amplitude']-a['amplitude'])/a['amplitude'],
                 d_res=resonance_distance(p['rho_final'], q_max=12)[0])
    curves += rows
    print(f"  pair {k:2d}-{k+1:<2d}  gap {(b['amplitude']-a['amplitude'])*LU:7.2f} km"
          f"  conv {sum(r['converged'] for r in rows)}/{len(rows)}   [{time.time()-t0:.0f}s]")

n_conv = sum(c['converged'] for c in curves)
print(f"\nTREATMENT: {n_conv}/{len(curves)} converged to genuine tori "
      f"({100*n_conv/len(curves):.0f}%)")

## 5. Internal validation — the displacement is interpolation error

If the probe is measuring something real rather than noise, the midpoint
displacement should scale as **gap^2**: linear interpolation across a curved family
has second-order error, and nothing else in the construction is quadratic.

Expect a log-log slope of 2.00. Also expect pairs 10-11 and 15-16 — nearly identical
absolute gaps (62.50 vs 62.47 km) at very different amplitudes (140 vs 452 km) — to
give nearly identical *raw* displacement, which they only do if the score tracks the
gap and not the amplitude.

A slope near 2 means the interpolant is essentially *on* a torus and the residual
distance is the family's own curvature, not a distance to anything physical.

In [ ]:
g_km = np.array([c['gap_km'] for c in curves])
raw_km = np.array([c['dist_max'] * c['amp_cand'] * LU for c in curves])
mid = np.abs(np.array([c['alpha'] for c in curves]) - 0.5) < 1e-9

slope, icpt = np.polyfit(np.log10(g_km[mid]), np.log10(raw_km[mid]), 1)
print(f"midpoint displacement vs gap: slope = {slope:.3f}   (2.000 = pure 2nd-order)")

fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.loglog(g_km[mid], raw_km[mid], 'o', ms=6, label='midpoint (alpha=0.5)')
xs = np.array([g_km.min(), g_km.max()])
ax.loglog(xs, 10**icpt * xs**slope, 'k--', lw=1,
          label=f'fit slope {slope:.3f}')
ax.set_xlabel('gap between adjacent members [km]')
ax.set_ylabel('curve displacement found by probe [km]')
ax.set_title('Probe score is second-order interpolation error')
ax.grid(alpha=.3, which='both'); ax.legend()
plt.tight_layout(); plt.show()

## 6. Bundle control — why the label is a distance, not a class

Displace a converged member along its own unstable bundle by a controlled amount and
ask what the probe returns. If the score is a genuine distance it should track the
displacement linearly.

**This is the experiment that decides C over B.** Watch the binary label flip between
d = 1e-8 and d = 1e-7 — a displacement of 1.6 m versus 16 m on a 202 km torus. The
underlying physics is smooth and monotone; the bit is an arbitrary cut across it.

**Also watch where linearity breaks.** Past ~1e-7 the solver stops finding the
*nearby* torus and slides to a different one, so `dist_max` is no longer a faithful
distance. That bounds the dynamic range of the regression target and is why the
dataset carries a censoring flag rather than pretending far-field distances are
measurements.

In [ ]:
m = famC[11]
DG = qpit.stability_matrix(m['U0'], m['T'], m['rho'], MU)
V, Ru = qpit.torus_bundle(m['U0'], m['T'], m['rho'], MU, 'unstable', DG=DG)
print(f"member A = {m['amplitude']*LU:.1f} km   R_u = {Ru:.3f}   nu = {qpit.stability_index(DG):.3f}")

bundle = []
for dd in (1e-8, 1e-7, 1e-6, 1e-5, 1e-4):
    p = probe_curve(m['U0'] + dd * V, m['T'], m['rho'], MU,
                    C_target=m['C'], offgrid=False)
    p.update(arm='bundle', displacement=dd,
             d_res=resonance_distance(p['rho_final'], q_max=12)[0])
    bundle.append(p)
    print(f"  d={dd:.0e}  conv={str(p['converged']):5s}  dist_max={p['dist_max']:.3e}"
          f"  raw={p['dist_max']*p['amp_cand']*LU:.3e} km  LABEL(B)={label_from_probe(p, cal)}")

dd = np.array([p['displacement'] for p in bundle])
dm = np.array([p['dist_max'] for p in bundle])
lin = dm < 1e-3                                     # linear regime
fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.loglog(dd[lin], dm[lin], 'o-', ms=6, label='faithful (linear regime)')
ax.loglog(dd[~lin], dm[~lin], 'x', ms=8, color='0.5', label='censored (solver slid)')
ax.axhline(cal['dist_max'], color='tab:red', ls='--', lw=1,
           label=f"B threshold {cal['dist_max']:.1e}")
ax.set_xlabel('displacement along unstable bundle [nondim]')
ax.set_ylabel('probe dist_max')
ax.set_title('The score is continuous; the binary is a cut across it')
ax.grid(alpha=.3, which='both'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. Gross negative

A 35% random curve perturbation. Not a serious control — it is a floor check that the
probe declines to invent tori. The control that matters is the resonance gap, which
needs a halo at a different energy and is the next notebook.

In [ ]:
rng = np.random.default_rng(3)
U_bad = m['U0'].copy()
U_bad[:, :3] += 0.35 * m['amplitude'] * rng.standard_normal((U_bad.shape[0], 3))
p_gross = probe_curve(U_bad, m['T'], m['rho'], MU, C_target=m['C'], offgrid=False)
p_gross.update(arm='gross', displacement=np.nan,
               d_res=resonance_distance(p_gross['rho_final'], q_max=12)[0])
print(f"converged={p_gross['converged']}  code={p_gross['code']}"
      f"  dist_max={p_gross['dist_max']:.3e}")

## 8. Dataset — two tables, and why

**Curve table**, one row per probe. **Point table**, one row per state.

The split is not bookkeeping. The label is genuinely per-curve: every theta on a
converged invariant curve shares it. Flattening to one table would hide that and
invite pseudo-replication — 8 thetas from one curve are 1 observation, not 8, and a
binomial CI over rows would be too narrow by sqrt(8).

`curve_id` is therefore the **group key**. Week 11 must split train/test on it, not
on rows, or points from the same curve land on both sides and the test set is
contaminated.

`censored` marks samples outside the probe's linear regime, where `dist_max` records
how far Newton travelled rather than how far the nearest torus is. Those are
lower bounds, not measurements.

In [ ]:
all_probes = curves + bundle + [p_gross]

CURVE_COLS = ['arm', 'alpha', 'pair_lo', 'pair_hi', 'gap_km', 'gap_rel',
              'displacement', 'converged', 'code', 'n_iter',
              'dist_max', 'dist_rms', 'damp_rel', 'res_inf',
              'C_target', 'C_seed', 'C_final', 'rho_final', 'amp_cand',
              'amp_final', 'reproj_rel', 'd_res']

CENSOR = 1e-3
curve_tbl = {c: np.array([p.get(c, np.nan) for p in all_probes],
                         dtype=object if c in ('arm', 'code') else float)
             for c in CURVE_COLS}
curve_tbl['censored'] = np.array([(not p['converged']) or p['dist_max'] > CENSOR
                                  for p in all_probes])
curve_tbl['label_B'] = np.array([bool(label_from_probe(p, cal)) for p in all_probes])
curve_tbl['score_C'] = curve_tbl['dist_max'].astype(float)

rng = np.random.default_rng(10)
N_THETA = 8
X, cid, thetas = [], [], []
for i, p in enumerate(all_probes):
    if not p['converged']:
        continue
    th = rng.uniform(0, 2*np.pi, N_THETA)
    X.append(qpit.evaluate_curve(p['result']['U0'], th))
    cid.append(np.full(N_THETA, i))
    thetas.append(th)
X = np.vstack(X); cid = np.concatenate(cid); thetas = np.concatenate(thetas)

print(f"curve table : {len(all_probes)} rows, {len(CURVE_COLS)+3} columns")
print(f"point table : {X.shape[0]} states from {len(set(cid))} curves")
print(f"censored    : {curve_tbl['censored'].sum()} / {len(all_probes)}")
print(f"label_B     : {curve_tbl['label_B'].sum()} positive")

## 9. Report B from C

Accuracy is a *function of tolerance*, not a number. Reporting the curve rather than a
point makes the threshold a stated dimension of the result instead of a hidden choice.

With only the gross negative available the curve is near-degenerate — the classes are
separated by three orders of magnitude and any cut between them scores perfectly. It
becomes informative once the resonance negatives exist, which is precisely the point
of building them.

In [ ]:
sc = curve_tbl['score_C'].astype(float)
is_pos = np.array([p['arm'] == 'gap' for p in all_probes])
is_neg = np.array([p['arm'] == 'gross' or
                   (p['arm'] == 'bundle' and p['displacement'] >= 1e-6)
                   for p in all_probes])

tols = np.logspace(-8, 1, 200)
acc = [((sc[is_pos] <= t).sum() + (sc[is_neg] > t).sum()) / (is_pos.sum()+is_neg.sum())
       for t in tols]

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.semilogx(tols, acc, lw=1.6)
ax.axvline(cal['dist_max'], color='tab:red', ls='--', lw=1,
           label=f"calibrated threshold {cal['dist_max']:.1e}")
ax.set_xlabel('tolerance on dist_max'); ax.set_ylabel('accuracy')
ax.set_ylim(0, 1.05); ax.grid(alpha=.3); ax.legend(fontsize=8)
ax.set_title('Accuracy is a curve, not a number')
plt.tight_layout(); plt.show()

print(f"accuracy at calibrated threshold: "
      f"{acc[int(np.searchsorted(tols, cal['dist_max']))]:.3f}")
print("NOTE: near-degenerate until resonance negatives exist.")

In [ ]:
np.savez('../data/qpit_dataset_w10_gap.npz',
         states=X, curve_id=cid, theta=thetas,
         score_C=curve_tbl['score_C'].astype(float),
         label_B=curve_tbl['label_B'],
         censored=curve_tbl['censored'],
         arm=np.array([str(a) for a in curve_tbl['arm']]),
         **{k: curve_tbl[k].astype(float) for k in CURVE_COLS
            if k not in ('arm', 'code')},
         threshold_dist_max=cal['dist_max'],
         threshold_damp_rel=cal['damp_rel'],
         threshold_res_inf=cal['res_inf'],
         censor_level=CENSOR, n_theta=N_THETA, mu=MU,
         alphas=np.array(ALPHAS))
print(f"saved  [{time.time()-t0:.0f}s total]")

## What this establishes, and what it does not

**Established.** 80/80 gap candidates lie on genuine invariant tori, across every gap
width the family contains. The naive labelling scheme — off my seventeen computed
surfaces, therefore negative — is wrong essentially everywhere in the gap, not merely
imperfect. The probe's score scales as gap^2 with a fitted exponent of 2.00, so it is
measuring interpolation error against a family that is locally solid.

**Not established.** Nothing here shows the probe declining to find a torus where one
genuinely should not exist. The gross negative is a strawman and the bundle control
finds tori because tori are genuinely nearby. That test requires the resonance gap.

**Next (still Week 10):** `nb10_resonance_control.ipynb` — halo family at
dC ~ +1.9e-3 to put rho_halo on 2/7, seed tori slightly above it, continue with
`pin='C'` and let kappa < 0 walk the family into the resonance until it terminates.
Candidates past termination are the negatives, and the approach itself is the
degradation signal Week 12 needs.

**Not this week:** feature representation, any training, hyperparameter work, the
phase-space failure map.